# YOLO PCB — Train trên Google Colab

Phiên bản Colab thay cho `setup_venv.sh` + `test_train.sh`.
Trên Colab **không cần `.venv`** (đã có sẵn Python, PyTorch và GPU). Notebook dùng lại đúng các script gốc:
`patch_data_yaml.py`, `detect_hardware.py`, `train.py`, `export_onnx.py`.

**Trước khi chạy:** `Runtime → Change runtime type → GPU` (khuyến nghị T4 / L4 / A100).

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi

## 2. Lấy mã nguồn + dataset (chọn **1 trong 2** cách)

Cần đưa cả thư mục `yolo` (gồm các file `.py`, `config.yaml`, `requirements.txt` và `yolo_dataset/`) lên Colab.

### Cách A — Google Drive (khuyến nghị)
Copy thư mục `yolo` vào Drive, ví dụ `MyDrive/yolo`, rồi chạy ô dưới.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Sửa lại cho đúng vị trí thư mục yolo trong Drive của bạn
PROJECT_DIR = '/content/drive/MyDrive/yolo'

%cd $PROJECT_DIR
!ls

### Cách B — Upload file zip
Nén thư mục `yolo` ở máy (`yolo.zip`), rồi chạy ô dưới và chọn file khi được hỏi.

In [ ]:
# from google.colab import files
# import zipfile, os
#
# uploaded = files.upload()  # chọn yolo.zip
# zip_name = next(iter(uploaded))
# with zipfile.ZipFile(zip_name) as z:
#     z.extractall('/content')
#
# PROJECT_DIR = '/content/yolo'  # sửa nếu giải nén ra tên khác
# %cd $PROJECT_DIR
# !ls

## 3. Cài đặt thư viện (thay cho `setup_venv`)

Colab đã có sẵn PyTorch bản CUDA nên **không cài lại torch**, chỉ cài các gói trong `requirements.txt`
(`ultralytics`, `pyyaml`, `psutil`, `onnx`, `onnxruntime`, `onnxslim`).

In [ ]:
!pip install -q -r requirements.txt

import torch
print('Torch:', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## 4. Patch `data.yaml` + dò phần cứng

Giống bước trong `setup_venv` / `train.sh`. `detect_hardware.py` sẽ tự ghi `config.local.yaml`
(chọn `device=0`, batch và cache phù hợp với GPU của Colab).

In [ ]:
!python patch_data_yaml.py
!python detect_hardware.py

## 5. Test train (tương đương `test_train.sh`)

Chạy nhanh **1 epoch, 1 model** để kiểm tra pipeline. `--device 0` dùng GPU.
Đổi `--epochs`, `--batch`, `--models` nếu muốn.

In [ ]:
!python train.py --models yolov8m.pt --epochs 1 --batch 4 --device 0

## 6. Test export ONNX

In [ ]:
!python export_onnx.py --models yolov8m.pt

## 7. Train đầy đủ (tuỳ chọn — tương đương `train.sh`)

Train tất cả model trong `config.yaml` (`yolov8m.pt` + `yolo26m.pt`, 100 epoch) rồi export ONNX.
Số epoch/batch lấy từ `config.yaml` và `config.local.yaml`.

In [ ]:
!python train.py
!python export_onnx.py

## 8. Tải kết quả về máy

Kết quả train nằm trong `runs/pcb/`, ONNX trong `runs/pcb/onnx/`.
Nếu dùng Google Drive (Cách A) thì kết quả đã tự lưu trong Drive. Ô dưới nén và tải về máy.

In [ ]:
import shutil
from google.colab import files

shutil.make_archive('runs_pcb', 'zip', 'runs/pcb')
files.download('runs_pcb.zip')